### Multiple Linear Regression & Multi-Feature Modeling
**Multiple Linear Regression** extends simple linear regression by fitting a linear relationship between multiple independent predictor features ($X_1, X_2, \dots, X_p$) and a continuous target ($y$).

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p$$

In matrix-vector form:
$$\hat{\mathbf{y}} = \mathbf{X} \boldsymbol{\beta}$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

In [ ]:
np.random.seed(2026)
n_samples = 150
experience_years = np.random.uniform(1, 15, size=n_samples)
education_level = np.random.choice([1, 2, 3], size=n_samples, p=[0.4, 0.4, 0.2])
certifications_count = np.random.poisson(lam=2, size=n_samples)
weekly_hours = np.random.normal(40, 5, size=n_samples)

# Target Salary Formula (True underlying function + Gaussian Noise)
# Salary = 35000 + 8500*(exp) + 12000*(edu) + 4500*(certs) + 150*(hours) + Noise
salary = (35000 
          + 8500 * experience_years 
          + 12000 * education_level 
          + 4500 * certifications_count 
          + 150 * weekly_hours 
          + np.random.normal(0, 5000, size=n_samples))

df_multi = pd.DataFrame({
    "experience_years": experience_years.round(1),
    "education_level": education_level,
    "certifications_count": certifications_count,
    "weekly_hours": weekly_hours.round(1),
    "salary": salary.round(2)
})

print("--- Multi-Feature Dataset Preview ---")
print(df_multi.head())

## 1. Matrix OLS Closed-Form Solution

The optimal weights $\boldsymbol{\beta}$ minimizing the Sum of Squared Residuals ($\text{SSR}$) are computed analytically via the Normal Equation:

$$\boldsymbol{\beta} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

*Note:* We must prepend a column of ones ($\mathbf{1}$) to the matrix $\mathbf{X}$ to account for the intercept term $\beta_0$.

In [ ]:
# 1. Prepare Feature Matrix X and Target Vector y
X_raw = df_multi[["experience_years", "education_level", "certifications_count", "weekly_hours"]].values
y_vec = df_multi["salary"].values

# why to add a bias column? 
# In linear regression, the bias term (intercept) allows the model to fit the data better by providing a baseline value when all features are zero.
# 2. Add Bias Column (intercept column of 1s)
X_design = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw])

# 3. Apply Normal Equation: (X^T * X)^(-1) * X^T * y
beta_matrix = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y_vec

feature_names = ["Intercept (Beta 0)", "experience_years", "education_level", "certifications_count", "weekly_hours"]

for name, coef in zip(feature_names, beta_matrix):
    print(f"{name:<<22}: ₹{coef:10.2f}")



In [ ]:
# same in Scikit-Learn


# Prepare Feature Matrix X and Target Vector y
X_df = df_multi[["experience_years", "education_level", "certifications_count", "weekly_hours"]]
y_df = df_multi["salary"]

# Initialize and fit the linear regression model
model = LinearRegression()
model.fit(X_df, y_df)

# Print the coefficients
print(f"Intercept (Beta 0): {model.intercept_:.2f}")
coef_df = pd.DataFrame({
    "Feature": X_df.columns,
    "Coefficient (₹)": model.coef_
}).sort_values(by="Coefficient (₹)", ascending=False)
print(coef_df.to_string(index=False))


In [18]:
print("Original Data")
# Copy 5 rows and drop the 'salary' column
# df = df_multi.drop(columns=["salary"]).copy().head(5)
df = df_multi[X_df.columns].copy().head(5)

# Now you can safely pass df directly into model.predict()
df["predicted_salary"] = model.predict(df)
print(df_multi.head())
print(df)


Original Data
   experience_years  education_level  certifications_count  weekly_hours  \
0               4.1                3                     4          40.3   
1               6.8                1                     4          36.5   
2              14.7                3                     2          41.1   
3               2.2                1                     1          35.5   
4               7.7                2                     1          36.0   

      salary  
0  129597.55  
1  130617.26  
2  211179.11  
3   70426.90  
4  136284.22  
   experience_years  education_level  certifications_count  weekly_hours  \
0               4.1                3                     4          40.3   
1               6.8                1                     4          36.5   
2              14.7                3                     2          41.1   
3               2.2                1                     1          35.5   
4               7.7                2                     1 

In [22]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
# VIF Calculation
# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X_df.columns

# Adding intercept column for statsmodels compatibility
X_vif = X_df.copy()
X_vif["intercept"] = 1.0

vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_df.shape[1])
]

print("--- Variance Inflation Factor (VIF) Analysis ---")
print(vif_data.round(2).to_string(index=False))

--- Variance Inflation Factor (VIF) Analysis ---
             Feature  VIF
    experience_years 1.04
     education_level 1.01
certifications_count 1.01
        weekly_hours 1.04
